# 🎲 Probability Theory for Machine Learning

Welcome to the world of uncertainty! ML models deal with randomness and incomplete information - that's where probability theory comes in.

## Why Probability?

- **Data is noisy** - measurements contain errors
- **Predictions are uncertain** - we need confidence intervals
- **Models are probabilistic** - from logistic regression to VAEs
- **Bayesian thinking** - update beliefs with evidence

## What You'll Learn
1. Probability fundamentals
2. Common probability distributions
3. Conditional probability and Bayes' theorem
4. Expected value and variance
5. Maximum likelihood estimation

In [ ]:
# Import our tools
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
from matplotlib.patches import Rectangle

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)
%matplotlib inline

## 1. Probability Basics

**Probability** measures how likely an event is:
- $P(A) = 0$: impossible
- $P(A) = 1$: certain
- $0 < P(A) < 1$: somewhere in between

### Key Rules:
1. **Sum rule**: $P(A \cup B) = P(A) + P(B) - P(A \cap B)$
2. **Product rule**: $P(A \cap B) = P(A|B) \cdot P(B)$
3. **Complement**: $P(\neg A) = 1 - P(A)$

In [ ]:
# Simulate coin flips
n_flips = 10000
flips = np.random.choice(['H', 'T'], size=n_flips)
heads_count = np.sum(flips == 'H')
prob_heads = heads_count / n_flips

# Cumulative probability
cumulative_heads = np.cumsum(flips == 'H')
cumulative_prob = cumulative_heads / np.arange(1, n_flips + 1)

# Visualize convergence to true probability
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Convergence plot
ax1.plot(cumulative_prob, linewidth=2, alpha=0.8)
ax1.axhline(y=0.5, color='r', linestyle='--', linewidth=2, label='True probability (0.5)')
ax1.set_xlabel('Number of flips', fontsize=12)
ax1.set_ylabel('Observed probability', fontsize=12)
ax1.set_title('Law of Large Numbers: Convergence to True Probability', 
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, n_flips)

# Histogram
ax2.bar(['Heads', 'Tails'], 
        [heads_count, n_flips - heads_count], 
        color=['#3498db', '#e74c3c'], 
        edgecolor='black', 
        linewidth=2)
ax2.axhline(y=n_flips/2, color='green', linestyle='--', linewidth=2, 
            label='Expected (5000)')
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title(f'Results from {n_flips} Coin Flips', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

for i, count in enumerate([heads_count, n_flips - heads_count]):
    ax2.text(i, count + 100, f'{count}\n({count/n_flips:.3f})', 
             ha='center', fontsize=12, fontweight='bold')

plt.suptitle('🪙 Understanding Probability Through Coin Flips', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print(f"After {n_flips} flips:")
print(f"Observed P(Heads) = {prob_heads:.4f}")
print(f"True P(Heads) = 0.5000")
print(f"Difference = {abs(prob_heads - 0.5):.4f}")

## 2. Probability Distributions

### Discrete Distributions

#### Bernoulli Distribution
Single binary outcome (coin flip, yes/no)
$$P(X=1) = p, \quad P(X=0) = 1-p$$

In [ ]:
# Bernoulli distribution
p = 0.7
samples = stats.bernoulli.rvs(p, size=1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# PMF
x = [0, 1]
pmf = [1-p, p]
ax1.bar(x, pmf, color=['#e74c3c', '#2ecc71'], 
        edgecolor='black', linewidth=2, width=0.5)
ax1.set_xticks([0, 1])
ax1.set_xticklabels(['Failure (0)', 'Success (1)'])
ax1.set_ylabel('Probability', fontsize=12)
ax1.set_title(f'Bernoulli PMF (p={p})', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 1)
ax1.grid(True, alpha=0.3, axis='y')

for i, prob in enumerate(pmf):
    ax1.text(i, prob + 0.03, f'{prob:.2f}', 
             ha='center', fontsize=13, fontweight='bold')

# Samples
unique, counts = np.unique(samples, return_counts=True)
ax2.bar(unique, counts, color=['#e74c3c', '#2ecc71'], 
        edgecolor='black', linewidth=2, width=0.5)
ax2.set_xticks([0, 1])
ax2.set_xticklabels(['Failure (0)', 'Success (1)'])
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title(f'1000 Samples from Bernoulli(p={p})', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

for i, count in zip(unique, counts):
    ax2.text(i, count + 10, f'{count}', ha='center', fontsize=13, fontweight='bold')

plt.suptitle('🎯 Bernoulli Distribution', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("Use case: Binary classification (spam/not spam, cat/dog)")

#### Binomial Distribution
Number of successes in $n$ independent Bernoulli trials
$$P(X=k) = \binom{n}{k} p^k (1-p)^{n-k}$$

In [ ]:
# Binomial distribution
n, p = 20, 0.5
x = np.arange(0, n+1)
pmf = stats.binom.pmf(x, n, p)

fig, ax = plt.subplots(figsize=(14, 7))
bars = ax.bar(x, pmf, color='skyblue', edgecolor='black', linewidth=1.5, alpha=0.8)

# Highlight the mean
mean = n * p
ax.axvline(x=mean, color='red', linestyle='--', linewidth=2.5, label=f'Mean = n·p = {mean}')

# Color the mean bar differently
bars[int(mean)].set_color('orange')
bars[int(mean)].set_edgecolor('red')
bars[int(mean)].set_linewidth(3)

ax.set_xlabel('Number of successes (k)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title(f'Binomial Distribution: n={n} trials, p={p}', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Mean: {mean}")
print(f"Variance: {n * p * (1-p)}")
print("\nUse case: Number of successful trials (e.g., number of heads in 20 coin flips)")

### Continuous Distributions

#### Normal (Gaussian) Distribution
The most important distribution in statistics!

$$f(x) = \frac{1}{\sqrt{2\pi\sigma^2}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

In [ ]:
# Different Gaussian distributions
x = np.linspace(-10, 10, 1000)

distributions = [
    (0, 1, 'Standard Normal μ=0, σ=1'),
    (0, 2, 'Wider: μ=0, σ=2'),
    (2, 1, 'Shifted: μ=2, σ=1'),
    (-2, 0.5, 'Narrow & Shifted: μ=-2, σ=0.5')
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, (mu, sigma, title) in enumerate(distributions):
    ax = axes[idx]
    pdf = stats.norm.pdf(x, mu, sigma)
    
    # Plot PDF
    ax.plot(x, pdf, 'b-', linewidth=3, label='PDF')
    ax.fill_between(x, pdf, alpha=0.3)
    
    # Mark mean
    ax.axvline(x=mu, color='red', linestyle='--', linewidth=2, label=f'Mean μ={mu}')
    
    # Mark standard deviations
    ax.axvline(x=mu-sigma, color='green', linestyle=':', linewidth=1.5, alpha=0.7)
    ax.axvline(x=mu+sigma, color='green', linestyle=':', linewidth=1.5, alpha=0.7, 
               label=f'±1σ')
    
    # Shade ±1σ region
    x_1sigma = x[(x >= mu-sigma) & (x <= mu+sigma)]
    pdf_1sigma = stats.norm.pdf(x_1sigma, mu, sigma)
    ax.fill_between(x_1sigma, pdf_1sigma, alpha=0.3, color='green')
    
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('Probability Density', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.1)
    
    # Add text showing 68% in ±1σ
    ax.text(mu, max(pdf)*0.5, '68%', fontsize=12, ha='center', 
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))

plt.suptitle('📊 Normal (Gaussian) Distribution', fontsize=17, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("Key properties of Normal distribution:")
print("• ~68% of data within μ ± 1σ")
print("• ~95% of data within μ ± 2σ")
print("• ~99.7% of data within μ ± 3σ")
print("\nUse case: Modeling errors, natural phenomena, Central Limit Theorem")

### Central Limit Theorem

**Mind-blowing fact**: Sum of many independent random variables → approximately Normal!

This is why Gaussians appear everywhere in ML.

In [ ]:
# Demonstrate Central Limit Theorem
n_samples = 1000
sample_sizes = [1, 2, 5, 30]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, n in enumerate(sample_sizes):
    # Generate samples from uniform distribution
    # Take mean of n samples
    means = []
    for _ in range(n_samples):
        sample = np.random.uniform(0, 1, n)
        means.append(np.mean(sample))
    means = np.array(means)
    
    ax = axes[idx]
    
    # Histogram
    ax.hist(means, bins=30, density=True, alpha=0.7, 
            color='skyblue', edgecolor='black', linewidth=1.5)
    
    # Overlay normal distribution
    mu, sigma = means.mean(), means.std()
    x = np.linspace(means.min(), means.max(), 100)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=3, 
            label=f'Normal fit\nμ={mu:.3f}, σ={sigma:.3f}')
    
    ax.set_xlabel('Sample mean', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'n = {n} (averaging {n} uniform samples)', 
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.suptitle('🎭 Central Limit Theorem: Uniform → Normal', 
             fontsize=17, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("Notice: As n increases, the distribution of sample means becomes more Normal!")
print("This works regardless of the original distribution.")

## 3. Conditional Probability & Bayes' Theorem

**Conditional probability**: What's the probability of A given that B occurred?

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

**Bayes' Theorem**: Update beliefs with evidence

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

Where:
- $P(A|B)$ = Posterior (updated belief)
- $P(B|A)$ = Likelihood (how likely is evidence given hypothesis)
- $P(A)$ = Prior (initial belief)
- $P(B)$ = Evidence (normalizing constant)

In [ ]:
# Medical test example
# Disease prevalence (prior)
P_disease = 0.01  # 1% of population has disease
P_no_disease = 1 - P_disease

# Test accuracy (likelihood)
P_positive_given_disease = 0.95  # Sensitivity: test detects disease 95% of time
P_positive_given_no_disease = 0.05  # False positive rate: 5%

# Using Bayes' theorem
# What's P(disease | positive test)?

# P(positive) = P(positive|disease)*P(disease) + P(positive|no disease)*P(no disease)
P_positive = (P_positive_given_disease * P_disease + 
              P_positive_given_no_disease * P_no_disease)

# Bayes' theorem
P_disease_given_positive = (P_positive_given_disease * P_disease) / P_positive

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Prior vs Posterior
categories = ['Prior\n(before test)', 'Posterior\n(after positive test)']
probabilities = [P_disease, P_disease_given_positive]
colors = ['#3498db', '#e74c3c']

bars = ax1.bar(categories, probabilities, color=colors, 
               edgecolor='black', linewidth=2, width=0.5)
ax1.set_ylabel('Probability of Disease', fontsize=12)
ax1.set_title('Bayesian Update: Prior → Posterior', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 0.25)
ax1.grid(True, alpha=0.3, axis='y')

for i, (cat, prob) in enumerate(zip(categories, probabilities)):
    ax1.text(i, prob + 0.01, f'{prob:.1%}', ha='center', 
             fontsize=14, fontweight='bold')

# Tree diagram
ax2.text(0.5, 0.9, 'Population', fontsize=13, ha='center', 
         fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))

# Disease branch
ax2.arrow(0.5, 0.85, -0.15, -0.15, head_width=0.02, head_length=0.03, 
          fc='red', ec='red', linewidth=2)
ax2.text(0.25, 0.68, f'Disease\n{P_disease:.1%}', fontsize=11, ha='center',
         bbox=dict(boxstyle='round', facecolor='#ffcccc', alpha=0.8))

ax2.arrow(0.25, 0.63, -0.08, -0.08, head_width=0.02, head_length=0.03, 
          fc='green', ec='green', linewidth=1.5)
ax2.text(0.12, 0.52, f'Test +\n{P_positive_given_disease:.0%}', 
         fontsize=10, ha='center',
         bbox=dict(boxstyle='round', facecolor='#ccffcc', alpha=0.8))

# No disease branch
ax2.arrow(0.5, 0.85, 0.15, -0.15, head_width=0.02, head_length=0.03, 
          fc='blue', ec='blue', linewidth=2)
ax2.text(0.75, 0.68, f'No Disease\n{P_no_disease:.0%}', fontsize=11, ha='center',
         bbox=dict(boxstyle='round', facecolor='#ccccff', alpha=0.8))

ax2.arrow(0.75, 0.63, 0.08, -0.08, head_width=0.02, head_length=0.03, 
          fc='orange', ec='orange', linewidth=1.5)
ax2.text(0.88, 0.52, f'Test +\n{P_positive_given_no_disease:.0%}', 
         fontsize=10, ha='center',
         bbox=dict(boxstyle='round', facecolor='#ffeecc', alpha=0.8))

# Result
ax2.text(0.5, 0.25, 
         f'Given positive test:\nP(Disease|+) = {P_disease_given_positive:.1%}', 
         fontsize=13, ha='center', fontweight='bold',
         bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.9, pad=1))

ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
ax2.axis('off')
ax2.set_title('Probability Tree', fontsize=13, fontweight='bold')

plt.suptitle('🔬 Bayes\' Theorem: Medical Test Example', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("=" * 60)
print("BAYES' THEOREM CALCULATION")
print("=" * 60)
print(f"Prior: P(Disease) = {P_disease:.1%}")
print(f"Likelihood: P(+|Disease) = {P_positive_given_disease:.0%}")
print(f"False positive rate: P(+|No Disease) = {P_positive_given_no_disease:.0%}")
print(f"\nPosterior: P(Disease|+) = {P_disease_given_positive:.1%}")
print("\nSurprising! Even with a positive test, only ~16% chance of having disease.")
print("This is because the disease is rare (low prior).")

## 4. Expected Value and Variance

**Expected value** (mean): Where the center is
$$E[X] = \sum x \cdot P(X=x) \quad \text{(discrete)}$$
$$E[X] = \int x \cdot f(x) dx \quad \text{(continuous)}$$

**Variance**: How spread out the distribution is
$$\text{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$

In [ ]:
# Compare distributions with same mean but different variances
x = np.linspace(-10, 10, 1000)
mu = 0

sigmas = [0.5, 1, 2, 3]
colors = ['red', 'orange', 'blue', 'purple']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# PDFs
for sigma, color in zip(sigmas, colors):
    pdf = stats.norm.pdf(x, mu, sigma)
    ax1.plot(x, pdf, linewidth=3, label=f'σ={sigma} (Var={sigma**2})', color=color)
    ax1.fill_between(x, pdf, alpha=0.2, color=color)

ax1.axvline(x=mu, color='black', linestyle='--', linewidth=2, label=f'Mean μ={mu}')
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('Same Mean, Different Variances', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Samples from each distribution
n_samples = 500
positions = np.arange(len(sigmas))

for i, (sigma, color) in enumerate(zip(sigmas, colors)):
    samples = np.random.normal(mu, sigma, n_samples)
    # Violin plot
    parts = ax2.violinplot([samples], positions=[i], widths=0.7,
                           showmeans=True, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.6)

ax2.set_xticks(positions)
ax2.set_xticklabels([f'σ={s}' for s in sigmas])
ax2.set_ylabel('Value', fontsize=12)
ax2.set_title('Samples from Each Distribution', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=mu, color='black', linestyle='--', linewidth=2, alpha=0.5)

plt.suptitle('📏 Expected Value vs Variance', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print("Key insight: Variance measures uncertainty!")
print("Higher variance = more spread out = more uncertain predictions")

## 5. Maximum Likelihood Estimation (MLE)

**Question**: Given data, what parameters best explain it?

**Answer**: Choose parameters that maximize the likelihood of observing the data!

$$\hat{\theta}_{MLE} = \arg\max_{\theta} P(\text{data}|\theta)$$

In [ ]:
# Generate data from unknown distribution
true_mu = 3.0
true_sigma = 1.5
data = np.random.normal(true_mu, true_sigma, 100)

# Try different mu values
mu_candidates = np.linspace(0, 6, 100)
log_likelihoods = []

for mu_test in mu_candidates:
    # Compute log-likelihood
    ll = np.sum(stats.norm.logpdf(data, mu_test, true_sigma))
    log_likelihoods.append(ll)

log_likelihoods = np.array(log_likelihoods)
best_mu = mu_candidates[np.argmax(log_likelihoods)]

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Data histogram
ax1.hist(data, bins=20, density=True, alpha=0.7, 
         color='skyblue', edgecolor='black', linewidth=1.5, label='Observed data')

# True distribution
x = np.linspace(data.min(), data.max(), 200)
ax1.plot(x, stats.norm.pdf(x, true_mu, true_sigma), 'r-', 
         linewidth=3, label=f'True: μ={true_mu}, σ={true_sigma}')

# MLE distribution
ax1.plot(x, stats.norm.pdf(x, best_mu, true_sigma), 'g--', 
         linewidth=3, label=f'MLE: μ={best_mu:.2f}')

ax1.axvline(x=true_mu, color='red', linestyle=':', linewidth=2, alpha=0.5)
ax1.axvline(x=best_mu, color='green', linestyle=':', linewidth=2, alpha=0.5)
ax1.set_xlabel('x', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Data & Fitted Distribution', fontsize=13, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Log-likelihood curve
ax2.plot(mu_candidates, log_likelihoods, 'b-', linewidth=3)
ax2.axvline(x=best_mu, color='green', linestyle='--', linewidth=2.5, 
            label=f'MLE: μ={best_mu:.2f}')
ax2.plot(best_mu, np.max(log_likelihoods), 'r*', markersize=20, 
         label='Maximum')
ax2.axvline(x=true_mu, color='red', linestyle=':', linewidth=2, 
            alpha=0.5, label=f'True: μ={true_mu}')
ax2.set_xlabel('μ (parameter)', fontsize=12)
ax2.set_ylabel('Log-Likelihood', fontsize=12)
ax2.set_title('Likelihood Function', fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.suptitle('🎯 Maximum Likelihood Estimation', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

print(f"True parameter: μ = {true_mu}")
print(f"MLE estimate: μ = {best_mu:.3f}")
print(f"Sample mean: μ = {np.mean(data):.3f}")
print("\nFor normal distribution, MLE = sample mean!")
print("This is the foundation of parameter estimation in ML.")

## 🎯 Practice Exercises

In [ ]:
# Exercise: Spam filter using Bayes' theorem
# P(spam) = 0.3
# P("free" | spam) = 0.8
# P("free" | not spam) = 0.1
# What's P(spam | "free")?

P_spam = 0.3
P_not_spam = 0.7
P_free_given_spam = 0.8
P_free_given_not_spam = 0.1

# TODO: Calculate P(spam | "free")
P_free = P_free_given_spam * P_spam + P_free_given_not_spam * P_not_spam
P_spam_given_free = (P_free_given_spam * P_spam) / P_free

print(f"P(spam | 'free') = {P_spam_given_free:.2%}")
print("\nIf an email contains 'free', it's likely spam!")

## 🚀 Key Takeaways

1. **Probability** quantifies uncertainty in ML predictions
2. **Distributions** model different types of random phenomena
3. **Normal distribution** appears everywhere (Central Limit Theorem)
4. **Bayes' theorem** updates beliefs with evidence
5. **Expected value** and **variance** characterize distributions
6. **MLE** finds parameters that best explain observed data

## Applications in ML

- **Classification**: Probabilities for each class
- **Regression**: Predict distribution, not just point estimate
- **Bayesian ML**: Update model with new data
- **Generative models**: Sample from learned distributions
- **Uncertainty quantification**: How confident are predictions?

## Next Steps

- Study multivariate distributions
- Learn about Bayesian inference
- Explore probabilistic graphical models
- Understand variational inference

### Resources
- "Probability Theory: The Logic of Science" by Jaynes
- "Pattern Recognition and Machine Learning" by Bishop
- Khan Academy: Probability & Statistics